# 06 — Evaluate and inspect mistakes
Reuse predictions for this checkpoint, or create them below. Run 05 first. Export every real validation box; score only labeled boxes.


**Padded-box experiment:** retains original DETR features and adds frozen local DINOv2 crop features. Uses this folder's caches, split snapshot, studies and checkpoints.


In [1]:
# Reuse missing base caches in this isolated experiment before checking prerequisites.
from config import reuse_base_caches
reuse_base_caches()

# Cell purpose: Load the original best checkpoint and decide whether its evaluation predictions are reusable.
# Inputs: Saved samples, runs/best.pt and any previous evaluation metadata.
# Outputs: A verified checkpoint/sample pairing and prediction-cache reuse decision.

from pathlib import Path
import json
from config import CONFIG, ROOT, fingerprint, save_json, require_cache, run_stage, model_files

# Edit config.py once. Restart the kernel after changing it.
# Shared paths come from config.py; they are not hard-coded separately in each notebook.
CACHE = CONFIG['cache_dir']
RUN = CONFIG['run_dir']
CACHE.mkdir(parents=True, exist_ok=True)
RUN.mkdir(parents=True, exist_ok=True)
print('Existing caches:', [p.name for p in CACHE.iterdir() if p.is_file()])

from IPython.display import display
import torch
import pandas as pd
from torch.utils.data import DataLoader
from model import BoxClassifier, collate, metrics, prediction_rows
# A missing prerequisite is created by its earlier notebook; an existing cache is reused.
require_cache(CACHE / 'samples.pt', '04_build_training_samples.ipynb')
# Refresh stale padded features before reusing samples/checkpoints.
from config import run_stage
run_stage('04_build_training_samples.ipynb')
assert (RUN / 'best.pt').exists(), 'Run notebook 05 first to train the classifier.'
data = torch.load(CACHE / 'samples.pt', map_location='cpu', weights_only=True)
checkpoint = torch.load(RUN / 'best.pt', map_location='cpu', weights_only=True)
# Reject a checkpoint whose sequence/features/split cache differs from the current samples.
assert checkpoint['samples_signature'] == data['signature'], 'Samples changed; rerun training'
PREDICTIONS = RUN / 'evaluation_predictions.json'
META = RUN / 'evaluation_meta.json'
import hashlib
# Tie reusable evaluation predictions to the exact saved checkpoint bytes.
checkpoint_hash = hashlib.sha256((RUN / 'best.pt').read_bytes()).hexdigest()
reuse = PREDICTIONS.exists() and META.exists() and json.loads(META.read_text())['checkpoint_sha256'] == checkpoint_hash
print('Reuse evaluated predictions:', reuse)

Existing caches: ['base_cache_provenance.json', 'base_samples.pt', 'deberta.pt', 'dino.pt', 'discovery.json', 'magi.pt', 'samples.pt', 'sequences.json']
Running prerequisite: 04_build_training_samples.ipynb
Running prerequisite: 03a_cache_dino_crops.ipynb
Local encoder: C:\MODELS\dinov2-small
Padding per side: 0.2 | matching feature cache: True
DINO features: C:\Users\ESHAAN\BYTE\Applied-Task\Magi\FINETUNING-EXPERMENTS\NarrationImbalanceTrainingPaddedBox\cache\dino.pt | sequences: 80
Reuse matching padded samples: True
Exact original split: 64 train / 16 validation
Original train class counts: [1142, 50]
Feature widths: text 768, DETR 256, DINO 384, geometry 4, tail 2; fused input 688.
Saved: C:\Users\ESHAAN\BYTE\Applied-Task\Magi\FINETUNING-EXPERMENTS\NarrationImbalanceTrainingPaddedBox\cache\samples.pt
Reuse evaluated predictions: False


Evaluate or reuse predictions, score labeled boxes and export review tables.


In [2]:
# Cell purpose: Evaluate or reuse predictions, score labeled boxes and export review tables.
# Inputs: The original validation split and best-checkpoint class probabilities.
# Outputs: Metrics JSON, all-real-box prediction CSV, labeled mistakes CSV and displayed summaries.

# Reuse predictions only after the checkpoint identity check has passed.
if reuse:
    rows = json.loads(PREDICTIONS.read_text(encoding='utf-8'))
else:
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = BoxClassifier(checkpoint['visual_dim'], checkpoint['max_positions'],
                          crop_dim=checkpoint['crop_dim']).to(device)
    model.load_state_dict(checkpoint['state_dict'])
    model.eval()
    val = [data['samples'][i] for i in data['val_indices']]
    loader = DataLoader(val, batch_size=CONFIG['batch_size'], shuffle=False, collate_fn=collate)
    rows = []
    # Disable autograd for frozen feature extraction or evaluation to reduce memory use.
    with torch.inference_mode():
        for batch in loader:
            batch = {k: v.to(device) if torch.is_tensor(v) else v for k, v in batch.items()}
            # Export real boxes only; prediction_rows excludes padding but retains ignored-label context boxes.
            rows.extend(prediction_rows(batch, model(batch).softmax(-1).cpu()))
    save_json(PREDICTIONS, rows)
    save_json(META, {'checkpoint_sha256': checkpoint_hash, 'epoch': checkpoint['epoch']})
report = metrics(rows)
# Exclude ignored real boxes from metrics/loss while retaining them in the full prediction export.
labeled = [r for r in rows if r['label'] != -100]
import math
weights = checkpoint['class_weights'].tolist()
report['val_loss'] = sum(-weights[r['label']] * math.log(max(
    r['p_spoken'] if r['label'] == 0 else r['p_narrated'], 1e-30)) for r in labeled) / len(labeled)
save_json(RUN / 'evaluation_metrics.json', report)
frame = pd.DataFrame(rows)
frame.to_csv(RUN / 'validation_predictions.csv', index=False)
# The mistake table includes only labeled boxes whose predicted class differs from GT.
errors = frame[(frame.label != -100) & (frame.label != frame.prediction)]
errors.to_csv(RUN / 'validation_errors.csv', index=False)
print(json.dumps(report, indent=2))
display(pd.crosstab(frame.loc[frame.label != -100, 'gt'], frame.loc[frame.label != -100, 'predicted_class']))
display(errors[['sequence_id', 'box_id', 'ocr', 'gt', 'predicted_class', 'p_spoken', 'p_narrated', 'tail_score']].head(30))

{
  "macro_f1": 0.754299175500589,
  "balanced_accuracy": 0.7919028340080971,
  "SPOKEN": {
    "precision": 0.9822064056939501,
    "recall": 0.968421052631579,
    "f1": 0.9752650176678446,
    "support": 285,
    "p_true_mean": 0.9588296611021322,
    "p_true_median": 0.9995905756950378,
    "p_true_gt_0.9": 0.9263157894736842,
    "p_true_lt_0.5": 0.031578947368421054
  },
  "NARRATED": {
    "precision": 0.47058823529411764,
    "recall": 0.6153846153846154,
    "f1": 0.5333333333333333,
    "support": 13,
    "p_true_mean": 0.6316799203363749,
    "p_true_median": 0.8834653496742249,
    "p_true_gt_0.9": 0.46153846153846156,
    "p_true_lt_0.5": 0.38461538461538464
  },
  "val_loss": 0.11194495518625568
}


predicted_class,NARRATED,SPOKEN
gt,,
NARRATED,8,5
SPOKEN,9,276


,sequence_id,box_id,ocr,gt,predicted_class,p_spoken,p_narrated,tail_score
296,seq_44b9a316b2919cfe,PG1:T7,"""Torso"" is estimated to be a dangerous A rate ...",NARRATED,SPOKEN,0.997171,0.002829,4.152107e-06
312,seq_44b9a316b2919cfe,PG3:T1,is taking my best efforts just to unify the team,SPOKEN,NARRATED,0.021920,0.978080,1.778734e-10
313,seq_44b9a316b2919cfe,PG3:T2,to say nothing of creating investigators like ...,SPOKEN,NARRATED,0.051936,0.948064,4.056475e-11
316,seq_44b9a316b2919cfe,PG3:T5,I UNDER- STAND that you're getting desperate,SPOKEN,NARRATED,0.000843,0.999156,1.675907e-03
317,seq_44b9a316b2919cfe,PG3:T6,Because Hirako's squad has a big lead on you,SPOKEN,NARRATED,0.131586,0.868414,2.621674e-03
341,seq_098e260f231aa789,PG2:T8,"Listen to me. You, all of you guys,",SPOKEN,NARRATED,0.021142,0.978858,3.742987e-04
370,seq_ac5793f8168efdd1,PG2:T9,"If we follow things like ""Article 13"",",SPOKEN,NARRATED,0.490444,0.509556,5.372515e-01
372,seq_ac5793f8168efdd1,PG2:T11,Rank 1 SASAKI.,SPOKEN,NARRATED,0.010440,0.989560,1.866098e-05
373,seq_ac5793f8168efdd1,PG2:T12,Hold on!,SPOKEN,NARRATED,0.067806,0.932194,2.649956e-02
376,seq_ac5793f8168efdd1,PG2:T15,"The quinx have ""experimen- Tal bodies"" so to s...",NARRATED,SPOKEN,0.687390,0.312610,1.487604e-03


Optional scratch cell; no operation is currently defined.


In [3]:
# Cell purpose: Optional scratch cell; no operation is currently defined.
# Inputs: None.
# Outputs: None; this cell contains comments only.

